# Convert Folkets lexicon pronunciations to JSON

> Ignoring everything else

- toc: false
- badges: true
- branch: master
- categories: [folkets, swedish, pronunciation, icu]

Based on [this]({% post_url 2025-10-24-convert-nst-lexicon %})

> Set up ICU

In [2]:
!pip install pyicu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.2/268.2 kB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyicu: filename=pyicu-2.16.2-cp312-cp312-linux_x86_64.whl size=2720236 sha256=75bcb316922d70a757ec24f7da5e599c765f919457d99e79885a65b2dba94a03
  Stored in directory: /root/.cache/pip/wheels/25/f3/cd/4923c874cedf8cdb8608035f48bb726fa040a98a66e2b13cea
Successfully built pyicu


> Get data

1.   [English-Swedish](https://folkets-lexikon.csc.kth.se/folkets/folkets_en_sv_public.xml)
2.   [Swedish-English](https://folkets-lexikon.csc.kth.se/folkets/folkets_sv_en_public.xml)


Example entry:

```xml
<word class="pp" comment="endast vid sifferuttryck" lang="sv" value="à"><translation comment="used only with numerical expressions" value="at" />
<phonetic soundFile="à.swf" value="a" />
<see type="saldo" value="à||à..1||à..pp.1" />
<example value="två koppar kaffe à 8 kronor (styck)"><translation value="two cups of coffee at 8 kronor (each)" />
</example>
<definition value="till ett pris av"><translation value="at a price of" />
</definition>
</word>
```

In [1]:
!wget https://folkets-lexikon.csc.kth.se/folkets/folkets_sv_en_public.xml

--2026-07-25 15:21:43--  https://folkets-lexikon.csc.kth.se/folkets/folkets_sv_en_public.xml
Resolving folkets-lexikon.csc.kth.se (folkets-lexikon.csc.kth.se)... 130.237.227.95
Connecting to folkets-lexikon.csc.kth.se (folkets-lexikon.csc.kth.se)|130.237.227.95|:443... connected.
HTTP request sent, awaiting response... 200 
Length: 14619005 (14M) [application/xml]
Saving to: ‘folkets_sv_en_public.xml’

folkets_sv_en_publi 100%[===================>]  13.94M  9.57MB/s    in 1.5s    

2026-07-25 15:21:45 (9.57 MB/s) - ‘folkets_sv_en_public.xml’ saved [14619005/14619005]



In [10]:
from lxml import etree

tree = etree.parse("folkets_sv_en_public.xml")

audio = {}
phon = {}
compounds = {}

for entry in tree.iter("word"):
    value = entry.get("value")
    word = value
    if not isinstance(word, str):
        word = str(entry)
    phonetic = entry.find("phonetic")
    if phonetic is None:
        continue
    if not isinstance(phonetic, etree._Element):
        print(phonetic)
        continue
    if "|" in value:
        compound = value.split("|")
        word = value.replace("|", "")
        if word in compounds:
            if compound == compounds[word]:
                continue
            else:
                print("Error:", compound, compounds[word])
        compounds[word] = compound

    phon_value = phonetic.get("value")
    if phon_value:
        if not entry in phon:
            phon[word] = set()
        phon[word].add(phon_value)
    sound_file = phonetic.get("soundFile")
    if sound_file:
        if not entry in audio:
            audio[word] = set()
        audio[word].add(sound_file)

In [13]:
phon

{'à': {'a'},
 'a': {'a:'},
 'a conto': {'akÅn:to'},
 'à jour': {'a$O:r'},
 'à la': {'ala'},
 'à la carte': {'alakAr+t:'},
 'AB': {'²A:be:'},
 'AB Svensk Bilprovning': {'sven:sk 2bI:lpro:vni@'},
 'AB Svenska Spel': {'²A:be: sven:ska spe:l'},
 'abbé': {'abE:'},
 'abbedissa': {'²abedIs:a'},
 'abborre': {'²Ab:år:e'},
 'abc': {'a:be:sE:'},
 'abc-bok': {'²A:be:se:bo:k'},
 'ABC-stridsmedel': {'²a:be:sE:stritsme:del'},
 'abdikera': {'abdikE:rar'},
 'aber': {'A:ber'},
 'ABF': {'a:be:Ef:'},
 'abnorm': {'abnÅr:m'},
 'abonnemang': {'abånemA@:'},
 'abonnent': {'abånEn:t'},
 'abonnera': {'abånE:rar'},
 'abort': {'abÅr+t:'},
 'abortrådgivning': {'²abÅr+t:rå:dji:vni@'},
 'abrakadabra': {'a:brakadA:bra'},
 'abrupt': {'abrUp:t'},
 'absolut': {'absolU:t'},
 'absolutism': {'absolutIs:m'},
 'absolutist': {'absolutIs:t'},
 'absorbera': {'absårbE:rar'},
 'abstinens': {'abstinEn:s'},
 'abstrahera': {'abstrahE:rar'},
 'abstrakt': {'abstrAk:t'},
 'abstraktion': {'abstrak$O:n'},
 'absurd': {'absUr:d'},
 'acceler

> Set up transliterator

In [ ]:
TRANSLIT_SV = """
r \+ n → ɳ ;
r \+ s → ʂ ;
r \+ l → ɭ ;
r \+ t → ʈ ;
r \+ d → ɖ ;

A → ɑ ;
O → ɔ ;
I → ɪ ;
E \* U → e \u2040 ʊ ;
E → ɛ ;
U → ʊ ;
Y → ʏ ;
2 → ø ;
9 → ø ;
u 0 → ɵ ;
\@ → ŋ ;
'""' → ² ;
'"' → ˈ ;
\% → ˌ ;
\: → ː ;
\$ → \. ;
g → ɡ ;
s \\\' → ɕ ;
x \\\\ → ɧ ;
\* → \u2040 ;
"""

<>:2: SyntaxWarning: invalid escape sequence '\`'
<>:2: SyntaxWarning: invalid escape sequence '\`'
/tmp/ipython-input-4250576132.py:2: SyntaxWarning: invalid escape sequence '\`'
  n\` → ɳ ;


In [ ]:
NST_TRANSLIT = r"""
::XSampa-IPA;

\$ → \. ;
\? → ˀ;
\* → \u2040 ;
"""

In [ ]:
DA_TRANSLIT = r"""
\? → ˀ;
\_  → ' ';
::XSampa-IPA;
\* → \u2040 ;
\$ → \. ;
"""

In [ ]:
import icu
def transliterator_from_rules(name, rules):
    fromrules = icu.Transliterator.createFromRules(name, rules)
    icu.Transliterator.registerInstance(fromrules)
    return icu.Transliterator.createInstance(name)

In [ ]:
swelex_trans = transliterator_from_rules("swelex_trans", TRANSLIT_SV)

In [ ]:
nstlex_trans = {}
nstlex_trans["no"] = transliterator_from_rules("nst_trans", NST_TRANSLIT)
nstlex_trans["da"] = transliterator_from_rules("da_trans", DA_TRANSLIT)

In [ ]:
assert swelex_trans.transliterate('""bA:n`s`$%ma$man') == "²bɑːɳʂ.ˌma.man"
assert swelex_trans.transliterate('"b9r$mIN$ham') == "ˈbør.mɪŋ.ham"
assert swelex_trans.transliterate('"bI$rU') == "ˈbɪ.rʊ"
assert swelex_trans.transliterate('""bIsp$%go:$d`en') == "²bɪsp.ˌɡoː.ɖen"

assert swelex_trans.transliterate('"x\\A:l') == "ˈɧɑːl"
assert swelex_trans.transliterate("\"s'u:$lens") == "ˈɕuː.lens"
assert swelex_trans.transliterate('a$"lE*U$te$n`a') == 'a.ˈle⁀ʊ.te.ɳa'
assert swelex_trans.transliterate('"fu0l') == 'ˈfɵl'

In [ ]:
def collapse_available_fields(data):
    output = []
    for i in range(1, 10):
        if data[f"available_field{i}"] != "":
            output.append(data[f"available_field{i}"])
        del data[f"available_field{i}"]
    data["available_fields"] = output
    return data

In [ ]:
def collapse_transliterations(data, transliterator):
    output = []
    for i in range(1, 5):
        if data[f"transliteration{i}"] != "":
            tmp = {}
            tmp["transliteration"] = data[f"transliteration{i}"]
            tmp["ipa"] = transliterator.transliterate(data[f"transliteration{i}"])
            tmp["certainty"] = data[f"certainty_trans_{i}"]
            tmp["status"] = data[f"status_trans_{i}"]
            tmp["language_code"] = data[f"language_code_trans_{i}"]
            output.append(tmp)
        del data[f"transliteration{i}"]
        del data[f"certainty_trans_{i}"]
        del data[f"status_trans_{i}"]
        del data[f"language_code_trans_{i}"]
    data["transliterations"] = output
    return data

In [ ]:
import json
import io
with open("svlex.json", "w") as outf:
    swelexf = io.StringIO(data["sv"])
    swelex = csv.DictReader(swelexf, delimiter=';', fieldnames=field_names, quoting=csv.QUOTE_NONE)
    for row in swelex:
        row["decomp"] = [f for f in row["decomp"].split("+") if f != ""]
        row = collapse_available_fields(row)
        row = collapse_transliterations(row, swelex_trans)
        jsonstr = json.dumps(row)
        outf.write(jsonstr + "\n")

In [ ]:
data["da"] = data["da"].replace('\r', '')
for lang in ["no", "da"]:
    with open(f"{lang}lex.json", "w", newline='') as outf:
        swelexf = io.StringIO(data[lang])
        swelex = csv.DictReader(swelexf, delimiter=';', fieldnames=field_names, quoting=csv.QUOTE_NONE)
        for row in swelex:
            row["decomp"] = [f for f in row["decomp"].split("+") if f != ""]
            row = collapse_available_fields(row)
            row = collapse_transliterations(row, nstlex_trans[lang])
            jsonstr = json.dumps(row)
            outf.write(jsonstr + "\n")